In [1]:
import os, sys, wave, struct

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython

from copy import deepcopy
from math import ceil
from scipy.io.wavfile import write

from scipy.signal.windows import hann

from IPython.display import Audio

In [ ]:
def load_sound(file):
    return wave.open(file, 'rb')
    
def plot_sound(data, times, name='default_name', save=False):
    plt.figure(figsize=(30, 4))
    plt.fill_between(times, data)
    plt.xlim(times[0], times[-1])
    plt.xlabel('time (s)')
    plt.ylabel('amplitude')
    if save:
        plt.savefig(name+'.png', dpi=100)
    plt.show()



In [3]:
data_path = os.getcwd()
filename = 'Sons/guitare.wav'
sound = os.path.join(data_path, filename) 

In [4]:
wavefile = load_sound(sound)
print(wavefile.getparams())

_wave_params(nchannels=1, sampwidth=2, framerate=22050, nframes=168072, comptype='NONE', compname='not compressed')


In [5]:
Fs = int(wavefile.getframerate())
num_samples = int(wavefile.getnframes())
data = wavefile.readframes(num_samples)
data = struct.unpack('{n}h'.format(n=num_samples), data)
x = np.array(data)

In [6]:
Audio(x, rate=Fs)

# 1. Phasing
The recurrent input-output equation is given by: 
$$f(n)=x(n) +a(n).x(n-p)$$

And the transfert function is given by: 
$$H(z) = 1 +az^{-p}$$

### 1.1 The phasing effect function

In [ ]:
def phasing_effect(x, a, p = 60): 
    x_pad = np.concatenate([np.zeros(p, dtype=x.dtype), x[:-p]])
    y = x + a * x_pad
    return y


### 1.2 Testing different values

In [138]:
fa = 10
amax = 0.5
amin = 0.3
B = (amax + amin)/2
A = (amax - amin)/2

an = [2 * np.pi *fa*i/Fs for i in range(len(x))]
an = np.array(an)
an = np.sin(an)
an = an * A + B

In [139]:
y = phasing_effect(x,an)

In [140]:
Audio(y, rate=Fs)

In [62]:
fa = 10
amax = 2
amin = 0.1
B = (amax + amin)/2
A = (amax - amin)/2

an = [2 * np.pi *fa*i/Fs for i in range(len(x))]
an = np.array(an)
an = np.sin(an)
an = an * A + B

In [63]:
y = phasing_effect(x,an)

In [64]:
Audio(y, rate=Fs)

# 2. Flanger 
### 2.1 The reccurent equation
This effect is similar to the previous one, but the varying variable is the delay and not the gain. 
$$y(n) = x(n) + a.x(n-p(n))$$

### 2.2 The flanger effect function

In [70]:
def flanger(x, p, a=1): 
    
    maxp = int(np.ceil(np.max(p)))
    x_pad = np.concatenate([np.zeros(maxp, dtype=x.dtype), x])

    y = np.empty_like(x)
    N = len(x)
    
    for n in range(N):
        d = int(np.ceil(p[n]))  
        y[n] = x[n] + a * x_pad[n + maxp - d]

    return y


### 2.3 Computing $p_{min}$ and $p_{max}$


The transfer function of the system is
$$ H(z)=1+a z^{-p} $$

With $a=1$, we get
$$ H(e^{j\omega}) = 1 + e^{-j\omega p}$$ and $$|H(e^{j\omega})|=\sqrt{1+a^2+2a\cos(\omega p)} = \sqrt{2+2\cos(\omega p)}$$

It is equal to $0$ when
$$
\omega p = (2k+1)\pi,\; k\in\mathbb{Z}.
$$

Using $\omega = 2\pi f/F_s$, this gives the zero frequencies
$$
f_k = \frac{(2k+1)F_s}{2p}.
$$

Therefore, the lowest zero in the spectrum is for $k=0$:
$$
f_0=\frac{F_s}{2p}.
$$

We want $f_0=f_{200}=200\,\text{Hz}$, so
$$
200=\frac{F_s}{2p_{200}}
\quad\Longrightarrow\quad
p_{200}=\frac{F_s}{400}.
$$

As our $F_s=22050\text{Hz}$  we have
$$
p_{200}=\frac{F_s}{400}=\frac{22050}{400}=55.125 \approx 55\ \text{samples}.
$$

Let's take
$p_{\min}=55 \times 0.5 = 28$ and $p_{\max}=55 \times 1.5 = 82$ 


### 2.4 Testing different values

In [90]:
fp = 10
pmin = 28
pmax = 82
B = (pmax + pmin)/2 
A = (pmax - pmin)/2

pn = [2 * np.pi *fp*i/Fs for i in range(len(x))]
pn = np.array(pn)
pn = np.sin(pn)
pn = pn * A + B

In [91]:
y = flanger(x, pn)

In [92]:
Audio(y, rate=Fs)

In [93]:
fp = 10
pmin = 50
pmax = 60
B = (pmax + pmin)/2 
A = (pmax - pmin)/2

pn = [2 * np.pi *fp*i/Fs for i in range(len(x))]
pn = np.array(pn)
pn = np.sin(pn)
pn = pn * A + B

In [94]:
y = flanger(x, pn)
Audio(y, rate=Fs)

# 3. Artificial reverbation
## 3.1 Early echoes simulation
### 3.1.1 What can we  improve the realism of the model for a real room ?

1. **Frequency-dependent reverberation time (due to air absorption):** in real rooms, high frequencies are absorbed more strongly than low frequencies. This can be modeled by using frequency-dependent coefficients.  

2. **Wall absorption** instead of using only the distance attenuation $a(r)=1/r$, each reflection should be weighted by a reflection coefficient $\rho$. For a $k$-th order reflection, the total gain becomes the product of the coefficients of the walls hit along the path. 

3. **Source and microphone directivity:** real sources are not perfectly omnidirectional and microphones have polar patterns. The direct sound and each image-source contribution can be weighted by angular gains depending on the source/mic orientation, improving realism.


### 3.1.2 How many image sources would you have if you include second order echoes ?

#### **4 walls:**

If we consider only the four lateral walls (excluding the floor and the ceiling), we have 4 first-order image sources (one per wall).

<img src="sources_4w_1st_order.png" width="350">

For second-order reflections, we obtain 8 image sources. Indeed, we can have two successive reflections on parallel walls (up/down, down/up, left/right, right/left) or one reflection on a horizontal wall and one on a vertical wall (up/right, up/left, down/right, down/left), giving 8 sources.

<img src="sources_4w_2nd_order.png" width="700">

The total number of image sources up to second order is therefore 12.

<img src="sources_all.png" width="700">

#### **6 walls:**

If we consider all 6 walls (including the floor and the ceiling), we have 6 first-order image sources and 18 second-order image sources, i.e., 24 image sources up to second order.


We thank Noah Parker and his online image-source method visualizer https://www.acs.psu.edu/noahparker/acs/img-src/img-src.html for the illustrations. 

### 3.1.3 To your opinion, is it necessary to include higher order echoes ? 

We can see that the number of image sources increases very quickly with the reflection order. Therefore, it is not necessary to explicitly include higher-order echoes in the early-echoes module: as the order increases, echoes become much denser and perceptually merge into the late reverberation (many echoes arrive within a short time interval). 


## 3.2 Late reverberation: Schroeder reverberator

### 3.2.1


**Comb filters:**

We have
$$
\frac{20\log_{10}(g_i)}{m_iT} = -\frac{60}{T_r}.
$$

So,
$$ g_i = 10^{-\frac{3 m_i T}{T_r}} = 10^{-\frac{3 m_i}{F_s T_r}} $$


With $\tau_i = m_iT$ (in seconds) we have an equivalent expression
$$g_i = 10^{-\frac{3\tau_i}{T_r}}$$

With comb delays $\tau \in \{29.7,\,37.1,\,41.4,\,43.7\}\text{ ms}$ we obtain:
$$
g_{1}(T_r)=10^{-\frac{3\cdot 0.0297}{T_r}},\quad
g_{2}(T_r)=10^{-\frac{3\cdot 0.0371}{T_r}},\quad
g_{3}(T_r)=10^{-\frac{3\cdot 0.0414}{T_r}},\quad
g_{4}(T_r)=10^{-\frac{3\cdot 0.0437}{T_r}}.
$$

**All-pass filter cell:**
The approximation is
$$
T_r \approx \frac{7T}{1-g_i^{1/m_i}}.
$$
Solving for $g_i$ yields
$$g_i = \left(1 - \frac{7}{F_sT_r}\right)^{m_i}$$
Using $\tau_i = m_iT$ (so $m_i=\tau_iF_s$), we can also write
$$
g_i = \left(1 - \frac{7}{F_sT_r}\right)^{\tau_iF_s}.
$$

With all-pass delays $\tau \in \{96.83,\,32.92\}\text{ ms}$:
$$
g_{1}(T_r)=\left(1 - \frac{7}{F_sT_r}\right)^{0.09683F_s},\qquad
g_{2}(T_r)=\left(1 - \frac{7}{F_sT_r}\right)^{0.03292F_s}.
$$



### 3.2.2

In [95]:
def comb_gain(tau_s, Tr):
    return 10 ** (-3.0 * tau_s / Tr)

def allpass_gain(m, Fs, Tr):
    base = 1.0 - 7.0 / (Fs * Tr)
    base = max(base, 1e-9)
    return base ** m

def comb_filter(x, m, g):
    # y[n] = x[n] + g*y[n-m]
    y = np.zeros_like(x)
    for n in range(len(x)):
        y[n] = x[n] + (g * y[n-m] if n >= m else 0.0)
    return y

def allpass_filter(x, m, g):
    # y[n] = -g*x[n] + x[n-m] + g*y[n-m]
    y = np.zeros_like(x)
    for n in range(len(x)):
        x_del = x[n-m] if n >= m else 0.0
        y_del = y[n-m] if n >= m else 0.0
        y[n] = -g * x[n] + x_del + g * y_del
    return y

def schroeder_reverb(x, Fs, Tr,
                     comb_delays_ms=(29.7, 37.1, 41.4, 43.7),
                     ap_delays_ms=(96.83, 32.92)):

    #4 combs in parallel, then sum
    s = np.zeros_like(x)
    for tau in comb_delays_ms:
        m = int(round(tau * 1e-3 * Fs))
        tau = m / Fs
        g = comb_gain(tau, Tr)
        s += comb_filter(x, m, g)

    # 2 all-pass in series 
    y = s
    for tau in ap_delays_ms:
        m = int(round(tau * 1e-3 * Fs))
        g = allpass_gain(m, Fs, Tr)
        y = allpass_filter(y, m, g)

    return y


In [103]:
Tr = 0.1  #small room
y = schroeder_reverb(x, Fs, Tr)
Audio(y, rate=Fs)

In [101]:
Tr = 0.5  #large room
y = schroeder_reverb(x, Fs, Tr)
Audio(y, rate=Fs)

In [ ]:
Tr = 2  #cathedral
y = schroeder_reverb(x, Fs, Tr)
Audio(y, rate=Fs)

### 3.2.3

In [108]:
comb_delays_ms = [29.7, 37.1, 41.4, 43.7]
ap_delays_ms   = [96.83, 32.92]

def ms_to_samples(ms, Fs):
    return int(round(ms * 1e-3 * Fs))

m_comb_nom = [ms_to_samples(d, Fs) for d in comb_delays_ms]
m_ap_nom   = [ms_to_samples(d, Fs) for d in ap_delays_ms]

print(m_comb_nom)
print(m_ap_nom)

[655, 818, 913, 964]
[2135, 726]


In [ ]:
import math
math.gcd(726, 2135)  #coprime test 

1

In [123]:
def samples_to_ms(samples,Fs):
    return (samples/Fs) * 1000

In [128]:
#coprime
m_comb_nom = [223, 1000, 613, 811, 999]
comb_delays_ms = [samples_to_ms(m,Fs) for m in m_comb_nom ]
m_ap_nom = [229, 500]
ap_delays_ms = [samples_to_ms(m,Fs) for m in m_ap_nom ]

print(comb_delays_ms)
print(ap_delays_ms)

[10.113378684807257, 45.35147392290249, 27.800453514739228, 36.78004535147392, 45.30612244897959]
[10.38548752834467, 22.675736961451246]


In [129]:
Tr = 0.5 
y = schroeder_reverb(x, Fs, Tr, comb_delays_ms=comb_delays_ms,ap_delays_ms=ap_delays_ms)
Audio(y, rate=Fs)

In [136]:
#not coprime
m_comb_nom = [400, 350, 700, 800]
comb_delays_ms = [samples_to_ms(m,Fs) for m in m_comb_nom ]
m_ap_nom = [600, 900, 999]
ap_delays_ms = [samples_to_ms(m,Fs) for m in m_ap_nom ]

print(comb_delays_ms)
print(ap_delays_ms)

[18.140589569160998, 15.873015873015872, 31.746031746031743, 36.281179138321995]
[27.210884353741495, 40.816326530612244, 45.30612244897959]


In [137]:
Tr = 0.5 
y = schroeder_reverb(x, Fs, Tr, comb_delays_ms=comb_delays_ms,ap_delays_ms=ap_delays_ms)
Audio(y, rate=Fs)